In [8]:
%pip install -q sentence-transformers scikit-learn pandas numpy


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:

# import packages
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Load the datasets from the workspace
train_df = pd.read_csv('question_classificatrion/question_classification_dataset/train.csv')
test_df = pd.read_csv('question_classificatrion/question_classification_dataset/test.csv')

# Display the first few rows of each DataFrame to verify
print('Train DataFrame head:')
print(train_df.head())
print('\nTest DataFrame head:')
print(test_df.head())


Train DataFrame head:
   label-coarse  label-fine                                               text
0             0           0  How did serfdom develop in and then leave Russ...
1             1           1   What films featured the character Popeye Doyle ?
2             0           0  How can I find a list of celebrities ' real na...
3             1           2  What fowl grabs the spotlight after the Chines...
4             2           3                    What is the full form of .com ?

Test DataFrame head:
   label-coarse  label-fine                                      text
0             4          40      How far is it from Denver to Aspen ?
1             5          21  What county is Modesto , California in ?
2             3          12                         Who was Galileo ?
3             0           7                         What is an atom ?
4             4           8          When did Hawaii become a state ?


In [10]:
#EMBEDDING MODEL

# model_name = "all-MiniLM-L6-v2"
model_name = "paraphrase-MiniLM-L6-v2"

embedder = SentenceTransformer(model_name)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
#Create Embeddings

X_train_text = train_df["text"].tolist()
X_test_text = test_df["text"].tolist()

X_train = embedder.encode(X_train_text, convert_to_numpy=True, show_progress_bar=True)
X_test = embedder.encode(X_test_text, convert_to_numpy=True, show_progress_bar=True)

print("Embedding shape:", X_train.shape)

Batches:   0%|          | 0/171 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Embedding shape: (5452, 384)


In [12]:
#Coarse Label Classifer
y_train_coarse = train_df["label-coarse"]
y_test_coarse = test_df["label-coarse"]

coarse_clf = LogisticRegression(max_iter=2000)
coarse_clf.fit(X_train, y_train_coarse)

coarse_preds = coarse_clf.predict(X_test)

print("Coarse Accuracy:", accuracy_score(y_test_coarse, coarse_preds))
print(classification_report(y_test_coarse, coarse_preds))

Coarse Accuracy: 0.86
              precision    recall  f1-score   support

           0       0.83      0.89      0.86       138
           1       0.80      0.68      0.74        94
           2       0.64      0.78      0.70         9
           3       0.90      0.92      0.91        65
           4       0.95      0.93      0.94       113
           5       0.86      0.88      0.87        81

    accuracy                           0.86       500
   macro avg       0.83      0.85      0.83       500
weighted avg       0.86      0.86      0.86       500



In [13]:
#Fine Label Classifier

y_train_fine = train_df["label-fine"]
y_test_fine = test_df["label-fine"]

fine_clf = LogisticRegression(max_iter=3000)
fine_clf.fit(X_train, y_train_fine)

fine_preds = fine_clf.predict(X_test)

print("Fine Accuracy:", accuracy_score(y_test_fine, fine_preds))
print(classification_report(y_test_fine, fine_preds))

Fine Accuracy: 0.802
              precision    recall  f1-score   support

           0       0.40      1.00      0.57         2
           2       0.75      0.75      0.75        16
           3       0.60      0.75      0.67         8
           4       0.90      0.98      0.94        55
           5       1.00      0.67      0.80         6
           6       0.00      0.00      0.00         1
           7       0.90      0.91      0.91       123
           8       0.96      0.96      0.96        47
           9       0.83      0.83      0.83         6
          10       1.00      0.50      0.67         2
          11       0.80      0.57      0.67         7
          12       0.50      0.50      0.50        10
          13       0.53      0.89      0.67         9
          14       0.65      0.73      0.69        74
          17       1.00      0.75      0.86         4
          18       0.60      1.00      0.75         3
          19       1.00      0.90      0.95        10
      

/Users/nautica/Documents/michigan/Classes/engr100/M4/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/nautica/Documents/michigan/Classes/engr100/M4/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/nautica/Documents/michigan/Classes/engr100/M4/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter

In [14]:
# Smmarize Data

models_to_try = [
    "all-MiniLM-L6-v2",
    "paraphrase-MiniLM-L6-v2"
]

results = []

for model_name in models_to_try:
    print(f"\nRunning model: {model_name}")

    embedder = SentenceTransformer(model_name)
    X_train = embedder.encode(train_df["text"].tolist(), convert_to_numpy=True, show_progress_bar=True)
    X_test = embedder.encode(test_df["text"].tolist(), convert_to_numpy=True, show_progress_bar=True)

    # Coarse
    coarse_clf = LogisticRegression(max_iter=2000)
    coarse_clf.fit(X_train, train_df["label-coarse"])
    coarse_preds = coarse_clf.predict(X_test)
    coarse_acc = accuracy_score(test_df["label-coarse"], coarse_preds)

    # Fine
    fine_clf = LogisticRegression(max_iter=3000)
    fine_clf.fit(X_train, train_df["label-fine"])
    fine_preds = fine_clf.predict(X_test)
    fine_acc = accuracy_score(test_df["label-fine"], fine_preds)

    results.append({
        "model": model_name,
        "coarse_accuracy": coarse_acc,
        "fine_accuracy": fine_acc
    })

results_df = pd.DataFrame(results)
print(results_df)


Running model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/171 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/171 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

                     model  coarse_accuracy  fine_accuracy
0         all-MiniLM-L6-v2            0.912          0.718
1  paraphrase-MiniLM-L6-v2            0.860          0.802
